In [ ]:
import polars as pl
from transformers import AutoTokenizer

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
DATA = "../data/raw/tweets.csv"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
token_limit = tokenizer.model_max_length

In [ ]:
raw_df = pl.read_csv(DATA)
df = raw_df.with_columns(
    (pl.col("isRetweet").str.to_lowercase() == "t").alias("is_retweet"),
    (pl.col("isDeleted").str.to_lowercase() == "t").alias("is_deleted"),
    (pl.col("isFlagged").str.to_lowercase() == "t").alias("is_flagged"),
    pl.col("date").str.to_datetime("%Y-%m-%d %H:%M:%S"),
    (pl.col("retweets") + 1).log().alias("log_retweets"),
    (pl.col("favorites") + 1).log().alias("log_favorites"),
    pl.col("device").cast(pl.Categorical),
    pl.col("text")
    .str.to_lowercase()
    .alias("text_lower"),
    pl.col("text")
    .str.to_lowercase()
    .str.replace_all(r"[^\w\s]", "")
    .alias("text_lower_punctless")
).drop(["isRetweet", "isDeleted", "isFlagged"]).with_row_index()
df

In [ ]:
col = "device"
df[col].value_counts().sort(by="count", descending=True)

In [ ]:
(df["retweets"] + 1).log().describe()

In [ ]:
df = df.with_columns(
    pl.col("text")
    .map_elements(
        lambda s: len(tokenizer.encode(s, add_special_tokens=True)),
        return_dtype=pl.Int64
    ).alias("token_count")
)
df

In [ ]:
exceeded_df = df.filter(pl.col("token_count") > token_limit)
exceeded_df

In [ ]:
df.write_parquet("../data/processed/trump_cleaned.parquet")

In [ ]:
embeddings_df = pl.read_parquet("../data/processed/trump_embeddings.parquet")
embeddings_df

In [ ]:
embeddings_df.filter(pl.col("text").is_null())